# Whole Slide Image (WSI) Analysis Demo

This notebook walks through the full PathAI pipeline: WSI loading via OpenSlide, tile extraction, MIL classification with attention pooling, and attention heatmap visualization.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print('pathai wsi analysis pipeline')

## 1. WSI Loading Pipeline (OpenSlide)

In [ ]:
# Note: actual WSI loading requires openslide-python and a .svs/.ndpi/.mrxs file
# This cell describes the pipeline and simulates the data structure

print('WSI Loading Pipeline (OpenSlide)')
print('=' * 50)
print()
print('real usage:')
print('  import openslide')
print('  slide = openslide.OpenSlide("TCGA-XX-XXXX.svs")')
print('  print(slide.dimensions)        # (width, height) at level 0')
print('  print(slide.level_count)       # number of pyramid levels')
print('  print(slide.level_downsamples) # [1.0, 4.0, 16.0, 64.0]')
print()

# simulate slide metadata
slide_info = {
    'filename': 'TCGA-AB-1234-01Z-00-DX1.svs',
    'dimensions_level0': (98304, 74240),   # ~7 gigapixels
    'mpp': 0.2526,                          # microns per pixel at level 0
    'level_count': 4,
    'level_downsamples': [1.0, 4.0, 16.0, 64.0],
    'magnification': '40x',
    'stain': 'H&E',
    'tissue_percentage': 0.73,
}

print('slide metadata:')
for k, v in slide_info.items():
    print(f'  {k:<25} {v}')

# compute tile grid
TILE_SIZE = 256
THUMBNAIL_SIZE = (512, 387)  # level 3 thumbnail

W, H = slide_info['dimensions_level0']
n_tiles_x = W // TILE_SIZE
n_tiles_y = H // TILE_SIZE
print(f'\ntile grid: {n_tiles_x} x {n_tiles_y} = {n_tiles_x * n_tiles_y:,} tiles at 256x256')

## 2. Tile Extraction Grid on Synthetic Thumbnail

In [ ]:
def create_synthetic_thumbnail(size=(512, 387)):
    """create a realistic-looking H&E thumbnail using noise + blobs"""
    W, H = size
    img = np.ones((H, W, 3), dtype=np.float32) * np.array([0.95, 0.92, 0.94])  # slide background

    # tissue region (irregular ellipse)
    Y, X = np.ogrid[:H, :W]
    cx, cy = W * 0.5, H * 0.5
    tissue_mask = ((X - cx) / (W * 0.4))**2 + ((Y - cy) / (H * 0.42))**2 < 1

    # H&E staining: pink (eosin) base with purple (hematoxylin) nuclei
    np.random.seed(7)
    noise = np.random.randn(H, W)
    from scipy.ndimage import gaussian_filter
    try:
        from scipy.ndimage import gaussian_filter
        smooth_noise = gaussian_filter(noise, sigma=8)
    except ImportError:
        smooth_noise = noise

    smooth_noise = (smooth_noise - smooth_noise.min()) / (smooth_noise.max() - smooth_noise.min())

    img[tissue_mask, 0] = 0.78 + 0.12 * smooth_noise[tissue_mask]  # R
    img[tissue_mask, 1] = 0.55 + 0.10 * smooth_noise[tissue_mask]  # G
    img[tissue_mask, 2] = 0.72 + 0.10 * smooth_noise[tissue_mask]  # B

    # add some "darker" tumor-like regions
    for (cx_, cy_, rx, ry) in [(200, 150, 60, 40), (320, 220, 80, 55), (150, 280, 50, 35)]:
        region = ((X - cx_) / rx)**2 + ((Y - cy_) / ry)**2 < 1
        region &= tissue_mask
        img[region, 0] -= 0.15
        img[region, 1] -= 0.10
        img[region, 2] -= 0.05

    return np.clip(img, 0, 1), tissue_mask


thumbnail, tissue_mask = create_synthetic_thumbnail((512, 387))

# simulate tissue percentage filter: select tiles with > 50% tissue
THUMB_TILE = 32  # tile size in thumbnail pixels
valid_tiles = []
for ty in range(0, thumbnail.shape[0] - THUMB_TILE, THUMB_TILE):
    for tx in range(0, thumbnail.shape[1] - THUMB_TILE, THUMB_TILE):
        tile_mask = tissue_mask[ty:ty+THUMB_TILE, tx:tx+THUMB_TILE]
        tissue_pct = tile_mask.mean()
        if tissue_pct > 0.5:
            valid_tiles.append((tx, ty, tissue_pct))

print(f'total tile positions: {(thumbnail.shape[0] // THUMB_TILE) * (thumbnail.shape[1] // THUMB_TILE)}')
print(f'tiles with >50% tissue: {len(valid_tiles)}')
print(f'filtering ratio: {len(valid_tiles) / ((thumbnail.shape[0] // THUMB_TILE) * (thumbnail.shape[1] // THUMB_TILE)):.1%}')

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.imshow(thumbnail)
for (tx, ty, pct) in valid_tiles[:80]:
    rect = patches.Rectangle((tx, ty), THUMB_TILE, THUMB_TILE,
                               linewidth=0.4, edgecolor='lime', facecolor='none', alpha=0.6)
    ax.add_patch(rect)
ax.set_title(f'tile extraction grid ({len(valid_tiles)} tissue tiles, >50% tissue threshold)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. MIL Classification on Pre-extracted Features

In [ ]:
class AttentionMIL(nn.Module):
    """attention-based MIL (ABMIL) for WSI classification"""
    def __init__(self, feature_dim=1024, hidden_dim=256, num_classes=2):
        super().__init__()
        # feature projection
        self.proj = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.25)
        )
        # attention network
        self.attn_V = nn.Linear(hidden_dim, hidden_dim)
        self.attn_U = nn.Linear(hidden_dim, hidden_dim)
        self.attn_w = nn.Linear(hidden_dim, 1)
        # classifier head
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, bag_features):
        # bag_features: (N, feature_dim) where N = number of tiles
        H = self.proj(bag_features)                        # (N, hidden)
        A = self.attn_w(torch.tanh(self.attn_V(H)) * torch.sigmoid(self.attn_U(H)))  # (N, 1)
        A = F.softmax(A, dim=0)                            # (N, 1) normalized
        Z = (A.T @ H).squeeze(0)                           # (hidden,) aggregated
        logits = self.classifier(Z.unsqueeze(0))           # (1, num_classes)
        return logits, A.squeeze(1)


# simulate pre-extracted features (e.g. from CONCH/UNI/PLIP encoder)
N_TILES = len(valid_tiles)
features = torch.randn(N_TILES, 1024)  # (N, 1024) bag of instances

# add signal: tiles in tumor regions get slightly different features
tumor_tile_indices = [i for i, (tx, ty, _) in enumerate(valid_tiles)
                      if 150 < tx < 380 and 120 < ty < 280]
features[tumor_tile_indices] += 2.0 * torch.randn(len(tumor_tile_indices), 1024) * 0.15

model = AttentionMIL(feature_dim=1024, hidden_dim=256, num_classes=2).eval()
print(f'ABMIL model params: {sum(p.numel() for p in model.parameters()):,}')

with torch.no_grad():
    logits, attn_scores = model(features)

probs = F.softmax(logits, dim=1)
pred_class = logits.argmax(dim=1).item()
class_names = ['benign', 'malignant']

print(f'bag size: {N_TILES} tiles')
print(f'prediction: {class_names[pred_class]} (logit={logits[0, pred_class]:.3f})')
print(f'probabilities: benign={probs[0,0]:.3f}, malignant={probs[0,1]:.3f}')
print(f'attention score range: [{attn_scores.min():.5f}, {attn_scores.max():.5f}]')

## 4. Attention Heatmap on Thumbnail

In [ ]:
# build attention heatmap on thumbnail
attn_np = attn_scores.cpu().numpy()

# normalize to [0, 1]
attn_norm = (attn_np - attn_np.min()) / (attn_np.max() - attn_np.min() + 1e-8)

heatmap = np.zeros((thumbnail.shape[0], thumbnail.shape[1]))
for i, (tx, ty, _) in enumerate(valid_tiles):
    heatmap[ty:ty+THUMB_TILE, tx:tx+THUMB_TILE] = attn_norm[i]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(thumbnail)
axes[0].set_title('H&E WSI thumbnail')
axes[0].axis('off')

axes[1].imshow(heatmap, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('attention heatmap')
axes[1].axis('off')
plt.colorbar(plt.cm.ScalarMappable(cmap='hot'), ax=axes[1], fraction=0.046, label='attention score')

# overlay
axes[2].imshow(thumbnail)
overlay = axes[2].imshow(heatmap, cmap='hot', alpha=0.45, vmin=0, vmax=1)
axes[2].set_title('overlay: attention on H&E')
axes[2].axis('off')
plt.colorbar(overlay, ax=axes[2], fraction=0.046, label='attention')

plt.suptitle(f'ABMIL attention heatmap — pred: {class_names[pred_class]} ({probs[0,pred_class]:.2%})', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Top-5 Highest Attention Tiles

In [ ]:
# get top-5 tiles by attention score
top5_indices = np.argsort(attn_norm)[::-1][:5]
top5_tiles = [(valid_tiles[i], attn_norm[i]) for i in top5_indices]

print('top-5 highest attention tiles:')
print(f'{"Rank":<6} {"Tile (tx,ty)":<20} {"Tissue %":<12} {"Attn Score"}')
print('-' * 55)
for rank, ((tx, ty, tissue_pct), attn_val) in enumerate(top5_tiles, 1):
    print(f'{rank:<6} ({tx:>3}, {ty:>3})            {tissue_pct:>6.1%}    {attn_val:.5f}')

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for rank, (ax, ((tx, ty, tissue_pct), attn_val)) in enumerate(zip(axes, top5_tiles), 1):
    # extract tile from thumbnail
    tile_img = thumbnail[ty:ty+THUMB_TILE, tx:tx+THUMB_TILE]
    ax.imshow(tile_img)
    ax.set_title(f'#{rank}\nattn={attn_val:.4f}\n({tx},{ty})', fontsize=8)
    # highlight border
    for spine in ax.spines.values():
        spine.set_edgecolor('red')
        spine.set_linewidth(2.5)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('top-5 highest attention tiles (most predictive regions)', fontsize=11)
plt.tight_layout()
plt.show()

# also show their locations on the thumbnail
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(thumbnail)
for rank, ((tx, ty, tissue_pct), attn_val) in enumerate(top5_tiles, 1):
    rect = patches.Rectangle((tx, ty), THUMB_TILE, THUMB_TILE,
                               linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.text(tx + 2, ty + THUMB_TILE - 5, f'#{rank}', color='white', fontsize=8,
            fontweight='bold', bbox=dict(boxstyle='round,pad=0.1', facecolor='red', alpha=0.7))
ax.set_title('top-5 attention tile locations on WSI thumbnail')
ax.axis('off')
plt.tight_layout()
plt.show()